In [3]:
# ============================================================
# Football Player Role Classification
# Kaggle Notebook — Full Pipeline
#
# Dataset: Football Players Detection (Roboflow Universe)
# Source: https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc
# License: CC BY 4.0
#
# Author: Marcelo Augusto
# Course: Pattern Recognition — MSc Software Engineering
# University of Europe for Applied Sciences
# ============================================================
 
# ============================================================
# CELL 1 — Install dependencies and download dataset
# ============================================================

# Install Roboflow to download the dataset programmatically
!pip install roboflow --quiet

from roboflow import Roboflow
import os

# Download the dataset from Roboflow in YOLOv8 format
# The dataset contains 4 classes: ball, goalkeeper, player, referee
rf = Roboflow(api_key="iVuN7A0WxJWl76ZYp0YN")
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(1)
dataset = version.download("yolov8", location="/kaggle/working/data", overwrite=True)

print("Dataset downloaded!")
print(f"Location: {dataset.location}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 5.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 103.8 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/data in yolov8:: 100%|██████████| 1338/1338 [00:00<00:00, 3274.83it/s]


Dataset downloaded!
Location: /kaggle/working/data


In [4]:
# ============================================================
# CELL 2 — Imports and configuration
# ============================================================
 
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import cv2
 
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

# ============================================================
# CONFIG
# ============================================================
DATA_DIR    = Path("/kaggle/working/data")    # raw YOLO dataset
DATASET_DIR = Path("/kaggle/working/dataset")          # extracted crops
BALANCED_DIR = Path("/kaggle/working/dataset_balanced") # balanced dataset
OUTPUT_DIR  = Path("/kaggle/working/outputs")          # models and figures
OUTPUT_DIR.mkdir(exist_ok=True)
 
IMG_SIZE   = 224
BATCH_SIZE = 32
SEED       = 42
CLASSES    = ['ball', 'goalkeeper', 'player', 'referee']
MAX_PER_CLASS = 1500  # max images per class after balancing

2026-06-06 08:25:33.830185: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780734334.030335      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780734334.086254      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780734334.548277      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780734334.548326      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780734334.548329      58 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [5]:
# ============================================================
# CELL 3 — Extract crops from YOLO dataset
# ============================================================
 
"""
The raw dataset uses YOLO format: each .txt label file contains
one line per object with: class_id cx cy width height (normalized).
We extract each annotated object as an individual crop and save it
into a folder named after its class — converting detection to classification.
"""
 
import cv2
 
MIN_CROP_SIZE = 10  # ignore bounding boxes smaller than this (pixels)
 
# Create output class folders
for cls in CLASSES:
    (DATASET_DIR / cls).mkdir(parents=True, exist_ok=True)
 
counts = {cls: 0 for cls in CLASSES}
 
for split in ['train', 'valid', 'test']:
    images_dir = DATA_DIR / split / 'images'
    labels_dir = DATA_DIR / split / 'labels'
 
    if not images_dir.exists():
        print(f"Skipping '{split}' — not found.")
        continue
 
    image_files = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png'))
    print(f"Processing '{split}': {len(image_files)} images")
 
    for img_path in image_files:
        label_path = labels_dir / (img_path.stem + '.txt')
        if not label_path.exists():
            continue
 
        img = cv2.imread(str(img_path))
        if img is None:
            continue
 
        h, w = img.shape[:2]
 
        with open(label_path) as f:
            lines = f.readlines()
 
        for i, line in enumerate(lines):
            parts = line.strip().split()
            if len(parts) != 5:
                continue
 
            cls_id = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
 
            # Convert YOLO normalized coords to pixel coordinates
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
 
            # Clamp to image boundaries
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
 
            if (x2 - x1) < MIN_CROP_SIZE or (y2 - y1) < MIN_CROP_SIZE:
                continue
 
            crop = img[y1:y2, x1:x2]
            cls_name = CLASSES[cls_id]
            out_path = DATASET_DIR / cls_name / f"{img_path.stem}_{i}.jpg"
            cv2.imwrite(str(out_path), crop)
            counts[cls_name] += 1
 
print("\nExtraction Summary:")
for cls, count in counts.items():
    print(f"  {cls:12s}: {count:>6} crops")
print(f"  {'TOTAL':12s}: {sum(counts.values()):>6} crops")

Processing 'train': 612 images
Processing 'valid': 38 images
Processing 'test': 13 images

Extraction Summary:
  ball        :    405 crops
  goalkeeper  :    473 crops
  player      :  13239 crops
  referee     :   1519 crops
  TOTAL       :  15636 crops


In [6]:
# ============================================================
# CELL 4 — Balance the dataset
# ============================================================
 
"""
The extraction results in heavy class imbalance:
  ball:       ~405 images
  goalkeeper: ~473 images
  player:    ~13239 images
  referee:   ~1519 images
 
We randomly sample up to MAX_PER_CLASS images per class to reduce bias.
"""
 
import shutil
import random
 
random.seed(SEED)
 
for cls in CLASSES:
    src = DATASET_DIR / cls
    dst = BALANCED_DIR / cls
    dst.mkdir(parents=True, exist_ok=True)
 
    images = list(src.glob('*.jpg'))
    selected = random.sample(images, min(MAX_PER_CLASS, len(images)))
 
    for img in selected:
        shutil.copy(img, dst / img.name)
 
    print(f"{cls:12s}: {len(selected):>6} images")
 
total = sum(len(list((BALANCED_DIR / cls).glob('*.jpg'))) for cls in CLASSES)
print(f"\nTotal balanced dataset: {total} images")

ball        :    405 images
goalkeeper  :    473 images
player      :   1500 images
referee     :   1500 images

Total balanced dataset: 3878 images


In [7]:
# ============================================================
# CELL 5 — Dataset visualization
# ============================================================
 
# Class distribution bar chart
counts_balanced = {cls: len(list((BALANCED_DIR / cls).glob('*.jpg'))) for cls in CLASSES}
 
plt.figure(figsize=(8, 5))
plt.bar(counts_balanced.keys(), counts_balanced.values(),
        color=['#2196F3', '#FF9800', '#4CAF50', '#E91E63'])
plt.title('Class Distribution — Balanced Dataset', fontsize=14, fontweight='bold')
plt.ylabel('Number of Images')
for i, (k, v) in enumerate(counts_balanced.items()):
    plt.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.pdf', format='pdf', bbox_inches='tight')
plt.show()
print("Saved: class_distribution.pdf")
 
# Sample images grid — 4 images per class
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, cls in enumerate(CLASSES):
    cls_files = list((BALANCED_DIR / cls).glob('*.jpg'))[:4]
    for j, img_path in enumerate(cls_files):
        img = keras.utils.load_img(str(img_path), target_size=(224, 224))
        img_array = keras.utils.img_to_array(img) / 255.0
        axes[i, j].imshow(img_array)
        axes[i, j].set_title(cls if j == 0 else '', fontsize=11, fontweight='bold')
        axes[i, j].axis('off')
plt.suptitle('Dataset Sample Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_images.pdf', format='pdf', bbox_inches='tight')
plt.show()
print("Saved: sample_images.pdf")

Saved: class_distribution.pdf
Saved: sample_images.pdf


In [8]:
# ============================================================
# CELL 6 — Data generators
# ============================================================
 
# Training generator with data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)
 
# Validation generator — only rescaling, no augmentation
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)
 
train_generator = train_datagen.flow_from_directory(
    str(BALANCED_DIR),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED,
    shuffle=True
)
 
val_generator = val_datagen.flow_from_directory(
    str(BALANCED_DIR),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False
)
 
print(f"Classes: {train_generator.class_indices}")
print(f"Train: {train_generator.samples} | Val: {val_generator.samples}")
 
# Compute class weights to handle residual imbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))
print(f"Class weights: {class_weight_dict}")

Found 3103 images belonging to 4 classes.
Found 775 images belonging to 4 classes.
Classes: {'ball': 0, 'goalkeeper': 1, 'player': 2, 'referee': 3}
Train: 3103 | Val: 775
Class weights: {0: np.float64(2.39429012345679), 1: np.float64(2.046833773087071), 2: np.float64(0.6464583333333334), 3: np.float64(0.6464583333333334)}


In [9]:
# ============================================================
# CELL 7 — Helper functions
# ============================================================
 
def get_callbacks(model_name):
    """EarlyStopping + ReduceLR + ModelCheckpoint callbacks."""
    return [
        EarlyStopping(patience=8, restore_best_weights=True, monitor='val_accuracy'),
        ReduceLROnPlateau(patience=4, factor=0.3, min_lr=1e-7, monitor='val_loss'),
        ModelCheckpoint(
            str(OUTPUT_DIR / f'best_{model_name}.keras'),
            save_best_only=True,
            monitor='val_accuracy'
        )
    ]
 
def plot_history(history, model_name):
    """Plot and save training/validation accuracy and loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
 
    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title(f'{model_name} — Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)
 
    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title(f'{model_name} — Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)
 
    plt.suptitle(model_name, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'curves_{model_name}.pdf', format='pdf', bbox_inches='tight')
    plt.show()
    print(f"Saved: curves_{model_name}.pdf")
 
def evaluate_model(model, generator, model_name):
    """Evaluate model and save confusion matrix."""
    generator.reset()
    preds = model.predict(generator, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = generator.classes
    class_names = list(generator.class_indices.keys())
 
    print(f"\n{'='*50}")
    print(f"Results: {model_name}")
    print('='*50)
    print(classification_report(y_true, y_pred, target_names=class_names))
 
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix — {model_name}', fontweight='bold', fontsize=13)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'cm_{model_name}.pdf', format='pdf', bbox_inches='tight')
    plt.show()
    print(f"Saved: cm_{model_name}.pdf")
 
    return np.mean(y_pred == y_true)

In [10]:
# ============================================================
# CELL 8 — Model 1: Custom CNN
# ============================================================
 
"""
A 4-block convolutional network built from scratch.
Blocks use increasing filter sizes (32→64→128→256) to learn
progressively more complex visual features.
"""
 
print("Training Custom CNN...")
 
inp = keras.Input(shape=(224, 224, 3))
 
# Block 1 — low-level features (edges, textures)
x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(inp)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2,2)(x)
x = layers.Dropout(0.25)(x)
 
# Block 2 — mid-level features
x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2,2)(x)
x = layers.Dropout(0.25)(x)
 
# Block 3 — higher-level features
x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2,2)(x)
x = layers.Dropout(0.25)(x)
 
# Block 4 — high-level semantic features (used for Grad-CAM)
x = layers.Conv2D(256, (3,3), activation='relu', padding='same', name='last_conv')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2,2)(x)
x = layers.Dropout(0.25)(x)
 
# Classifier head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
out = layers.Dense(4, activation='softmax')(x)
 
custom_cnn = keras.Model(inp, out, name='Custom_CNN')
custom_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
custom_cnn.summary()
 
history_cnn = custom_cnn.fit(
    train_generator,
    epochs=50,
    validation_data=val_generator,
    callbacks=get_callbacks('custom_cnn'),
    class_weight=class_weight_dict,
    verbose=1
)
plot_history(history_cnn, 'Custom_CNN')

Training Custom CNN...


I0000 00:00:1780734360.390090      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780734360.396329      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "Custom_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ last_conv (Conv2D)              │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,02

 Total params: 458,180 (1.75 MB)

...

Epoch 1/50


I0000 00:00:1780734367.270749     154 service.cc:152] XLA service 0x7866b810b6e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780734367.270833     154 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780734367.270841     154 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780734368.190686     154 cuda_dnn.cc:529] Loaded cuDNN version 91002


 2/97 ━━━━━━━━━━━━━━━━━━━━ 9s 95ms/step - accuracy: 0.2031 - loss: 2.5647  

I0000 00:00:1780734379.479213     154 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


97/97 ━━━━━━━━━━━━━━━━━━━━ 72s 575ms/step - accuracy: 0.4028 - loss: 1.6375 - val_accuracy: 0.3871 - val_loss: 1.6150 - learning_rate: 0.0010
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 43s 446ms/step - accuracy: 0.5208 - loss: 1.2056 - val_accuracy: 0.1484 - val_loss: 2.2354 - learning_rate: 0.0010
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 43s 441ms/step - accuracy: 0.5946 - loss: 1.0270 - val_accuracy: 0.1368 - val_loss: 3.4498 - learning_rate: 0.0010
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 44s 454ms/step - accuracy: 0.6226 - loss: 0.8965 - val_accuracy: 0.1045 - val_loss: 5.4347 - learning_rate: 0.0010
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 44s 449ms/step - accuracy: 0.6594 - loss: 0.7963 - val_accuracy: 0.1626 - val_loss: 3.5989 - learning_rate: 0.0010
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 44s 449ms/step - accuracy: 0.7180 - loss: 0.6019 - val_accuracy: 0.2090 - val_loss: 3.3519 - learning_rate: 3.0000e-04
Epoch 7/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 44s 450ms/step - accuracy: 0.7473 - loss: 0.5412 - 

In [11]:
# ============================================================
# CELL 9 — Model 2: MobileNetV2 (Transfer Learning)
# ============================================================
 
"""
MobileNetV2 pre-trained on ImageNet, fine-tuned in two phases:
  Phase 1 — Feature extraction: freeze base, train head only
  Phase 2 — Fine-tuning: unfreeze last 30 layers, train with low LR
"""
 
print("Training MobileNetV2...")
 
inputs_m = keras.Input(shape=(224, 224, 3))
base_m = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inputs_m)
base_m.trainable = False  # Phase 1: freeze base
 
x = base_m.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
out_m = layers.Dense(4, activation='softmax')(x)
mobilenet = keras.Model(inputs_m, out_m, name='MobileNetV2')
 
mobilenet.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
 
# Phase 1: train head only
history_mobilenet = mobilenet.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    callbacks=get_callbacks('mobilenet'),
    class_weight=class_weight_dict,
    verbose=1
)
 
# Phase 2: fine-tune — unfreeze last 30 layers
print("Fine-tuning MobileNetV2...")
base_m.trainable = True
for layer in base_m.layers[:-30]:
    layer.trainable = False
 
# Use a much smaller learning rate to preserve pre-trained weights
mobilenet.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
 
history_mobilenet_ft = mobilenet.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=get_callbacks('mobilenet_ft'),
    class_weight=class_weight_dict,
    verbose=1
)
plot_history(history_mobilenet_ft, 'MobileNetV2')

Training MobileNetV2...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/30


2026-06-06 08:43:39.265212: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-06 08:43:39.402230: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


30/97 ━━━━━━━━━━━━━━━━━━━━ 28s 425ms/step - accuracy: 0.4015 - loss: 1.3176

2026-06-06 08:44:06.622568: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-06 08:44:06.760394: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 579ms/step - accuracy: 0.4642 - loss: 1.0771

2026-06-06 08:44:54.518924: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-06 08:44:54.656133: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


97/97 ━━━━━━━━━━━━━━━━━━━━ 96s 780ms/step - accuracy: 0.5240 - loss: 0.8743 - val_accuracy: 0.6671 - val_loss: 0.7280 - learning_rate: 0.0010
Epoch 2/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 44s 450ms/step - accuracy: 0.6294 - loss: 0.6639 - val_accuracy: 0.6348 - val_loss: 0.7982 - learning_rate: 0.0010
Epoch 3/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 44s 457ms/step - accuracy: 0.6652 - loss: 0.6135 - val_accuracy: 0.7381 - val_loss: 0.6298 - learning_rate: 0.0010
Epoch 4/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 45s 460ms/step - accuracy: 0.6816 - loss: 0.5798 - val_accuracy: 0.7497 - val_loss: 0.5749 - learning_rate: 0.0010
Epoch 5/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 43s 447ms/step - accuracy: 0.6942 - loss: 0.5517 - val_accuracy: 0.7019 - val_loss: 0.6906 - learning_rate: 0.0010
Epoch 6/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 43s 448ms/step - accuracy: 0.7058 - loss: 0.5535 - val_accuracy: 0.6697 - val_loss: 0.7038 - learning_rate: 0.0010
Epoch 7/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 44s 450ms/step - accuracy: 0.7222 - loss: 0.5166 - val_

In [12]:
# ============================================================
# CELL 10 — Model 3: ResNet50 (Transfer Learning)
# ============================================================
 
"""
ResNet50 uses residual (skip) connections enabling training of deeper networks.
Same two-phase transfer learning strategy as MobileNetV2.
"""
 
print("Training ResNet50...")
 
inputs_r = keras.Input(shape=(224, 224, 3))
base_r = ResNet50(weights='imagenet', include_top=False, input_tensor=inputs_r)
base_r.trainable = False
 
x = base_r.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
out_r = layers.Dense(4, activation='softmax')(x)
resnet = keras.Model(inputs_r, out_r, name='ResNet50')
 
resnet.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
 
history_resnet = resnet.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    callbacks=get_callbacks('resnet'),
    class_weight=class_weight_dict,
    verbose=1
)
 
# Fine-tune last 15 layers
print("Fine-tuning ResNet50...")
base_r.trainable = True
for layer in base_r.layers[:-15]:
    layer.trainable = False
 
resnet.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
 
history_resnet_ft = resnet.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=get_callbacks('resnet_ft'),
    class_weight=class_weight_dict,
    verbose=1
)
plot_history(history_resnet_ft, 'ResNet50')

Training ResNet50...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 79s 644ms/step - accuracy: 0.2452 - loss: 1.4950 - val_accuracy: 0.1045 - val_loss: 1.3943 - learning_rate: 0.0010
Epoch 2/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 46s 474ms/step - accuracy: 0.2591 - loss: 1.3980 - val_accuracy: 0.1045 - val_loss: 1.4463 - learning_rate: 0.0010
Epoch 3/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 47s 484ms/step - accuracy: 0.2510 - loss: 1.3884 - val_accuracy: 0.3019 - val_loss: 1.3769 - learning_rate: 0.0010
Epoch 4/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 47s 482ms/step - accuracy: 0.2559 - loss: 1.3796 - val_accuracy: 0.4026 - val_loss: 1.3708 - learning_rate: 0.0010
Epoch 5/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 47s 486ms/step - accuracy: 0.2643 - loss: 1.3594 - val_accuracy: 0.5123 - val_loss: 1.2932 - learning_rate: 0.0010
Epoch 6/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 46s 472ms/step - accuracy: 0.2897 - loss: 1.3381 - val_accuracy: 0.3252 - val_loss: 1.3607 - learning_rate: 0.0010
Epoch 7/30

In [13]:
# ============================================================
# CELL 11 — Evaluate all models
# ============================================================
 
acc_cnn    = evaluate_model(custom_cnn, val_generator, 'Custom_CNN')
acc_mobile = evaluate_model(mobilenet,  val_generator, 'MobileNetV2')
acc_resnet = evaluate_model(resnet,     val_generator, 'ResNet50')
 
# Model comparison chart
models_names = ['Custom CNN', 'MobileNetV2', 'ResNet50']
accuracies   = [acc_cnn, acc_mobile, acc_resnet]
 
plt.figure(figsize=(8, 5))
bars = plt.bar(models_names, [a*100 for a in accuracies],
               color=['#2196F3', '#4CAF50', '#FF5722'])
plt.ylim(0, 100)
plt.title('Model Comparison — Validation Accuracy', fontweight='bold', fontsize=14)
plt.ylabel('Accuracy (%)')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 1,
             f'{acc*100:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_comparison.pdf', format='pdf', bbox_inches='tight')
plt.show()
print("Saved: model_comparison.pdf")
 
print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)
for name, acc in zip(models_names, accuracies):
    print(f"{name}: {acc*100:.2f}%")


Results: Custom_CNN
              precision    recall  f1-score   support

        ball       0.87      1.00      0.93        81
  goalkeeper       0.27      0.76      0.40        94
      player       0.81      0.39      0.53       300
     referee       0.89      0.81      0.85       300

    accuracy                           0.66       775
   macro avg       0.71      0.74      0.68       775
weighted avg       0.78      0.66      0.68       775

Saved: cm_Custom_CNN.pdf

Results: MobileNetV2
              precision    recall  f1-score   support

        ball       1.00      1.00      1.00        81
  goalkeeper       0.39      0.65      0.49        94
      player       0.85      0.47      0.61       300
     referee       0.76      0.93      0.84       300

    accuracy                           0.73       775
   macro avg       0.75      0.76      0.73       775
weighted avg       0.77      0.73      0.72       775

Saved: cm_MobileNetV2.pdf

Results: ResNet50
              pre

In [14]:
# ============================================================
# CELL 12 — Grad-CAM Visualizations
# ============================================================
 
"""
Grad-CAM (Gradient-weighted Class Activation Mapping) highlights
which image regions most influenced each model's prediction.
 
Steps:
1. Build sub-model outputting target conv layer + final predictions
2. Record gradients of predicted class score w.r.t. conv layer output
3. Pool gradients spatially to get per-channel importance weights
4. Compute weighted sum of feature maps → raw heatmap
5. Apply ReLU and normalize to [0, 1]
6. Overlay on original image using JET colormap
"""
 
def make_gradcam(img_exp, model, conv_layer_name):
    """Compute Grad-CAM heatmap for the predicted class."""
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_exp)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
 
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), predictions.numpy()[0]
 
def overlay_heatmap(img_array, heatmap):
    """Overlay JET heatmap on original image (60% image + 40% heatmap)."""
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    img_uint8 = np.uint8(img_array * 255)
    return cv2.addWeighted(img_uint8, 0.6, heatmap_colored, 0.4, 0)
 
def generate_gradcam_figure(model, conv_layer_name, model_name, sample_files, class_names):
    """Generate and save Grad-CAM figure for one model."""
    n = len(sample_files)
    fig, axes = plt.subplots(3, n, figsize=(n * 2.5, 9))
 
    for i, img_path in enumerate(sample_files):
        img = keras.utils.load_img(img_path, target_size=(224, 224))
        img_array = keras.utils.img_to_array(img) / 255.0
        img_exp = np.expand_dims(img_array, axis=0)
        true_class = Path(img_path).parent.name
 
        try:
            heatmap, preds = make_gradcam(img_exp, model, conv_layer_name)
            pred_class = class_names[np.argmax(preds)]
            confidence = np.max(preds)
            correct = true_class == pred_class
 
            # Row 0: original image
            axes[0, i].imshow(img_array)
            axes[0, i].set_title(
                f'True: {true_class}\nPred: {pred_class}\n{confidence:.0%}',
                fontsize=7, color='green' if correct else 'red'
            )
            axes[0, i].axis('off')
 
            # Row 1: raw heatmap
            axes[1, i].imshow(cv2.resize(heatmap, (224, 224)), cmap='jet', vmin=0, vmax=1)
            axes[1, i].set_title('Heatmap', fontsize=7)
            axes[1, i].axis('off')
 
            # Row 2: overlay
            axes[2, i].imshow(overlay_heatmap(img_array, heatmap))
            axes[2, i].set_title('Overlay', fontsize=7)
            axes[2, i].axis('off')
 
        except Exception as e:
            print(f"  Error: {e}")
            for row in range(3):
                axes[row, i].axis('off')
 
    for row, label in enumerate(['Original', 'Heatmap', 'Overlay']):
        axes[row, 0].set_ylabel(label, fontsize=9, fontweight='bold')
 
    plt.suptitle(f'Grad-CAM — {model_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'gradcam_{model_name}.pdf', format='pdf', bbox_inches='tight')
    plt.show()
    print(f"Saved: gradcam_{model_name}.pdf")
 
# Select 2 validation samples per class for visualization
sample_files = []
for cls in CLASSES:
    files = [f for f in val_generator.filepaths if f'/{cls}/' in f][:2]
    sample_files.extend(files)
 
class_names = list(val_generator.class_indices.keys())
 
# Generate Grad-CAM for all three models
models_config = [
    (custom_cnn, 'last_conv',           'Custom_CNN'),
    (mobilenet,  'Conv_1',              'MobileNetV2'),
    (resnet,     'conv5_block3_3_conv', 'ResNet50'),
]
 
for model, conv_layer, name in models_config:
    print(f"\nGenerating Grad-CAM for {name}...")
    generate_gradcam_figure(model, conv_layer, name, sample_files, class_names)
 
print("\nAll outputs saved to:", OUTPUT_DIR)
print("Pipeline complete!")


Generating Grad-CAM for Custom_CNN...
Saved: gradcam_Custom_CNN.pdf

Generating Grad-CAM for MobileNetV2...
Saved: gradcam_MobileNetV2.pdf

Generating Grad-CAM for ResNet50...
Saved: gradcam_ResNet50.pdf

All outputs saved to: /kaggle/working/outputs
Pipeline complete!
